# 05 — Evaluation

Load checkpoints, run inference, compute metrics.

## Setup

In [1]:
import sys, os, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
sys.path.insert(0, os.path.abspath(".."))

from utils.label_conversion import CLASSES, CLASS_TO_IDX, convert_segments
from utils.dataset import HarmonixDataset, MELSPEC_DIR, SEGMENT_DIR
from utils.spectnt import SpecTNT
from utils.postprocessing import postprocess
from utils.metrics import evaluate_all, evaluate_song
from utils.target_generation import build_targets, TARGET_FPS

device = torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
SAVE_DIR = "../checkpoints"


Device: cuda


## Load metadata

In [2]:
import pandas as pd
META_PATH = os.path.abspath("../data/harmonixset/dataset/metadata.csv")
meta = pd.read_csv(META_PATH, encoding="utf-8", encoding_errors="replace")
all_song_ids = meta["File"].tolist()
print(f"Total songs: {len(all_song_ids)}")


Total songs: 912


## 4-fold split

In [3]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=4, shuffle=True, random_state=42)
folds = list(kfold.split(all_song_ids))


## Inference function

In [4]:
def infer_song(model, sid, chunk_frames=130):
    """Run SpecTNT on a full song chunk-by-chunk, average overlapping regions."""
    melspec = np.load(os.path.join(MELSPEC_DIR, f"{sid}-mel.npy"))
    total_native = melspec.shape[1]
    total_target = total_native // 4

    all_b = np.zeros((total_target, 1), dtype=np.float32)
    all_f = np.zeros((total_target, 7), dtype=np.float32)
    counts = np.zeros(total_target, dtype=np.float32)

    hop = chunk_frames // 4
    for start in range(0, total_target - chunk_frames + 1, hop):
        native_start = start * 4
        native_end = native_start + chunk_frames * 4
        if native_end > total_native:
            break
        chunk = melspec[:, native_start:native_end]
        chunk_t = torch.from_numpy(chunk).unsqueeze(0).unsqueeze(0).float().to(device)
        with torch.no_grad():
            b, f = model(chunk_t)
        b = b.squeeze(0).cpu().numpy()
        f = f.squeeze(0).cpu().numpy()
        end = min(start + chunk_frames, total_target)
        seg_len = end - start
        all_b[start:end] += b[:seg_len]
        all_f[start:end] += f[:seg_len]
        counts[start:end] += 1

    mask = counts > 0
    all_b[mask] /= counts[mask, None]
    all_f[mask] /= counts[mask, None]
    return all_b, all_f


## Evaluate a variant across all folds

In [5]:
def evaluate_variant(variant="base"):
    all_pred_boundaries = []
    all_pred_labels = []
    all_ref_boundaries = []
    all_ref_labels = []

    for fold_idx, (_, val_idx) in enumerate(folds):
        ckpt_path = os.path.join(SAVE_DIR, f"spectnt_{variant}_fold{fold_idx+1}.pt")
        if not os.path.exists(ckpt_path):
            print(f"  No checkpoint for fold {fold_idx+1}, skipping")
            continue

        model = SpecTNT().to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
        model.load_state_dict(state_dict)
        model.eval()
        print(f"Fold {fold_idx+1}: loaded checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")

        val_ids = [all_song_ids[i] for i in val_idx]
        for sid in tqdm(val_ids, desc=f"  Evaluating fold {fold_idx+1}"):
            try:
                b_logits, f_logits = infer_song(model, sid)
                boundaries_pred, labels_pred, _ = postprocess(
                    b_logits.squeeze(-1), f_logits, fps=TARGET_FPS,
                )

                seg_path = os.path.join(SEGMENT_DIR, f"{sid}.txt")
                boundaries_ref = []
                labels_ref = []
                with open(seg_path, encoding="utf-8", errors="replace") as f:
                    for line in f:
                        parts = line.strip().split(maxsplit=1)
                        if len(parts) == 2:
                            boundaries_ref.append(float(parts[0]))
                            labels_ref.append(parts[1].strip())
                labels_ref = [l for l in labels_ref if l != "end"]

                conv_labels, _ = convert_segments(boundaries_ref[:-1], labels_ref)

                all_pred_boundaries.append(boundaries_pred)
                all_pred_labels.append(labels_pred)
                all_ref_boundaries.append(boundaries_ref)
                all_ref_labels.append(conv_labels)
            except Exception as e:
                print(f"  Error on {sid}: {e}")

    results = evaluate_all(
        all_pred_boundaries, all_pred_labels,
        all_ref_boundaries, all_ref_labels,
    )
    return results


## Run evaluation (requires trained checkpoints)

In [6]:
# Uncomment after training is complete:
print("=== SpecTNT (no CTL) ===")
results_base = evaluate_variant("base")
for k, v in results_base.items():
    print(f"  {k}: {v:.4f}")

print("\n=== SpecTNT (with CTL) ===")
results_ctl = evaluate_variant("ctl")
for k, v in results_ctl.items():
    print(f"  {k}: {v:.4f}")


=== SpecTNT (no CTL) ===
Fold 1: loaded checkpoint (epoch 9, val_loss=0.5148)


  Evaluating fold 1: 100%|██████████| 228/228 [01:40<00:00,  2.28it/s]


Fold 2: loaded checkpoint (epoch 7, val_loss=0.5157)


  Evaluating fold 2: 100%|██████████| 228/228 [01:32<00:00,  2.47it/s]


Fold 3: loaded checkpoint (epoch 6, val_loss=0.5287)


...

Fold 3: loaded checkpoint (epoch 9, val_loss=0.6945)


  Evaluating fold 3: 100%|██████████| 228/228 [01:15<00:00,  3.01it/s]


Fold 4: loaded checkpoint (epoch 7, val_loss=0.7065)


  Evaluating fold 4: 100%|██████████| 228/228 [01:33<00:00,  2.44it/s]


  hr.5f: 0.2748
  pwf: 0.5344
  sf: 0.1970
  acc: 0.4577
  chr.5f: 0.1604
  cf1: 0.6574
  macro_f1: 0.1422


## Smoothing Comparison

Compare baseline peak-picking + argmax vs. Viterbi-smoothed labels.
Transition matrix is estimated from training-set label bigrams.

In [7]:
from utils.smoothing import compute_transition_matrix, smoothed_postprocess
from utils.dataset import SEGMENT_DIR
from scipy.stats import wilcoxon

def evaluate_variant_smoothed(variant="base"):
    all_pred_boundaries = []
    all_pred_labels = []
    all_ref_boundaries = []
    all_ref_labels = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        ckpt_path = os.path.join(SAVE_DIR, f"spectnt_{variant}_fold{fold_idx+1}.pt")
        if not os.path.exists(ckpt_path):
            print(f"  No checkpoint for fold {fold_idx+1}, skipping")
            continue

        train_ids = [all_song_ids[i] for i in train_idx]
        trans_mat = compute_transition_matrix(train_ids, SEGMENT_DIR, self_loop_prior=50.0)

        model = SpecTNT().to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
        model.load_state_dict(state_dict)
        model.eval()

        val_ids = [all_song_ids[i] for i in val_idx]
        for sid in tqdm(val_ids, desc=f"  Smoothed fold {fold_idx+1}"):
            try:
                b_logits, f_logits = infer_song(model, sid)
                boundaries_pred, labels_pred, _ = smoothed_postprocess(
                    b_logits.squeeze(-1), f_logits, trans_mat, fps=TARGET_FPS,
                )

                seg_path = os.path.join(SEGMENT_DIR, f"{sid}.txt")
                boundaries_ref = []
                labels_ref = []
                with open(seg_path, encoding="utf-8", errors="replace") as f:
                    for line in f:
                        parts = line.strip().split(maxsplit=1)
                        if len(parts) == 2:
                            boundaries_ref.append(float(parts[0]))
                            labels_ref.append(parts[1].strip())
                labels_ref = [l for l in labels_ref if l != "end"]

                conv_labels, _ = convert_segments(boundaries_ref[:-1], labels_ref)

                all_pred_boundaries.append(boundaries_pred)
                all_pred_labels.append(labels_pred)
                all_ref_boundaries.append(boundaries_ref)
                all_ref_labels.append(conv_labels)
            except Exception as e:
                print(f"  Error on {sid}: {e}")

    results = evaluate_all(
        all_pred_boundaries, all_pred_labels,
        all_ref_boundaries, all_ref_labels,
    )
    return results


In [8]:
def compare_smoothing(variant="base"):
    """Run baseline + smoothed and print comparison table."""
    print(f"=== {variant} baseline vs. smoothed ===")
    ref = evaluate_variant(variant)
    smo = evaluate_variant_smoothed(variant)

    print(f"  {'Metric':<12} {'Baseline':<10} {'Smoothed':<10} {'Change':<10}")
    print(f"  {'-'*42}")
    for k in ref:
        diff = smo[k] - ref[k]
        print(f"  {k:<12} {ref[k]:<10.4f} {smo[k]:<10.4f} {diff:<+10.4f}")
    return ref, smo

# Uncomment below to run comparison (takes ~15 min per variant):
print("\\n" + "="*50)
ref_base, smo_base = compare_smoothing("base")
print("\\n" + "="*50)
ref_ctl, smo_ctl = compare_smoothing("ctl")


\n==================================================
=== base baseline vs. smoothed ===
Fold 1: loaded checkpoint (epoch 9, val_loss=0.5148)


  Evaluating fold 1: 100%|██████████| 228/228 [01:32<00:00,  2.47it/s]


Fold 2: loaded checkpoint (epoch 7, val_loss=0.5157)


  Evaluating fold 2: 100%|██████████| 228/228 [01:22<00:00,  2.76it/s]


Fold 3: loaded checkpoint (epoch 6, val_loss=0.5287)


...

Fold 3: loaded checkpoint (epoch 9, val_loss=0.6945)


  Evaluating fold 3: 100%|██████████| 228/228 [00:56<00:00,  4.03it/s]


Fold 4: loaded checkpoint (epoch 7, val_loss=0.7065)


  Smoothed fold 4: 100%|██████████| 228/228 [00:59<00:00,  3.84it/s]


  Metric       Baseline   Smoothed   Change    
  ------------------------------------------
  hr.5f        0.2748     0.2748     +0.0000   
  pwf          0.5344     0.5212     -0.0131   
  sf           0.1970     0.1674     -0.0295   
  acc          0.4577     0.4580     +0.0004   
  chr.5f       0.1604     0.1560     -0.0044   
  cf1          0.6574     0.6515     -0.0059   
  macro_f1     0.1422     0.1352     -0.0070   
